In [1]:
import pandas as pd
import numpy as np
import os
import re
import string
import nltk
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup, BertModel, DataCollatorWithPadding
from torch.utils.data import DataLoader, Dataset, random_split
import torch
from tqdm import tqdm
import logging
import torch.nn as nn
import torch.optim as optim
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
from transformers import LongformerForSequenceClassification
from transformers import LongformerTokenizer
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import json

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
keywords = []
with open('/content/drive/My Drive/EHR_PROJ/DATA/mimic_keywords.json', 'r') as f:
    data = json.load(f)
    for key, value in data.items():
        keywords.extend(value)

In [6]:
!pip install datasets
import datasets
dataset = datasets.load_dataset('ucberkeley-dlab/measuring-hate-speech')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 13.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.03k [00:00<?, ?B/s]

measuring-hate-speech.parquet:   0%|          | 0.00/14.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/135556 [00:00<?, ? examples/s]

In [83]:
df = dataset['train'].to_pandas()

In [84]:
cols = ['hate_speech_score', 'text', 'platform']

In [85]:
hate_data = df[cols]

In [86]:
# prompt: If hate_speech_socre greater than 0.5, make it 1. If it is less than -1, make it 0. Remove the rest

# Apply the conditions to the 'hate_speech_score' column
hate_data['hate_speech_binary'] = hate_data['hate_speech_score'].apply(lambda x: 1 if x > 0.5 else (0 if x < -1 else np.nan))

# Remove rows with NaN values (those that didn't meet either condition)
hate_data = hate_data.dropna(subset=['hate_speech_binary'])

# Display the updated DataFrame (optional)
hate_data

<ipython-input-86-54ceeb79741d>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hate_data['hate_speech_binary'] = hate_data['hate_speech_score'].apply(lambda x: 1 if x > 0.5 else (0 if x < -1 else np.nan))


,hate_speech_score,text,platform,hate_speech_binary
0,-3.90,Yes indeed. She sort of reminds me of the elde...,3,0.0
1,-6.52,The trans women reading this tweet right now i...,2,0.0
4,1.54,For starters bend over the one in pink and kic...,0,1.0
5,-4.93,Sounds like the kinda wholsesome life I'd die ...,0,0.0
7,2.08,Fuck off you insufferable retarded faggot.,0,1.0
...,...,...,...,...
135550,-2.49,@AbeShinzo @realDonaldTrump @shinzoabe 独裁者は行きま...,2,0.0
135551,-4.88,عاجل سماحة #السيد_عبدالملك_بدرالدين_الحوثي نص...,2,0.0
135552,-4.40,Millions of #Yemen-is participated in mass ral...,2,0.0
135553,-2.49,@AbeShinzo @realDonaldTrump @shinzoabe 独裁者は行きま...,2,0.0


In [87]:
hate_data['hate_speech_binary'].value_counts()

,count
hate_speech_binary,
0.0,53651
1.0,49048


In [88]:
import re

def preprocess_text(text):
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Remove mentions (@)
    text = re.sub(r'@\S+', '', text)
    # Remove hashtags
    text = re.sub(r'#\S+', '', text)
    # Remove non-English words (basic example, refine as needed)
    text = re.sub(r'[^\x00-\x7F]+', '', text)

    return text

In [89]:
hate_data['processed_text'] = hate_data['text'].apply(preprocess_text)

In [90]:
hate_data

,hate_speech_score,text,platform,hate_speech_binary,processed_text
0,-3.90,Yes indeed. She sort of reminds me of the elde...,3,0.0,Yes indeed. She sort of reminds me of the elde...
1,-6.52,The trans women reading this tweet right now i...,2,0.0,The trans women reading this tweet right now i...
4,1.54,For starters bend over the one in pink and kic...,0,1.0,For starters bend over the one in pink and kic...
5,-4.93,Sounds like the kinda wholsesome life I'd die ...,0,0.0,Sounds like the kinda wholsesome life I'd die ...
7,2.08,Fuck off you insufferable retarded faggot.,0,1.0,Fuck off you insufferable retarded faggot.
...,...,...,...,...,...
135550,-2.49,@AbeShinzo @realDonaldTrump @shinzoabe 独裁者は行きま...,2,0.0,Dictator goes This is the people of Iran w...
135551,-4.88,عاجل سماحة #السيد_عبدالملك_بدرالدين_الحوثي نص...,2,0.0,|Sayyed Abdulmalik:The Saudi r...
135552,-4.40,Millions of #Yemen-is participated in mass ral...,2,0.0,Millions of participated in mass rallies on 1...
135553,-2.49,@AbeShinzo @realDonaldTrump @shinzoabe 独裁者は行きま...,2,0.0,Dictator goes This is the people of Iran w...


In [91]:

# Keep only the 'hate_speech_binary' and 'processed_text' columns
hate_data = hate_data[['hate_speech_binary', 'processed_text']]

# Display the updated DataFrame (optional)
hate_data

,hate_speech_binary,processed_text
0,0.0,Yes indeed. She sort of reminds me of the elde...
1,0.0,The trans women reading this tweet right now i...
4,1.0,For starters bend over the one in pink and kic...
5,0.0,Sounds like the kinda wholsesome life I'd die ...
7,1.0,Fuck off you insufferable retarded faggot.
...,...,...
135550,0.0,Dictator goes This is the people of Iran w...
135551,0.0,|Sayyed Abdulmalik:The Saudi r...
135552,0.0,Millions of participated in mass rallies on 1...
135553,0.0,Dictator goes This is the people of Iran w...


In [97]:
# Assuming 'hate_data' is your DataFrame
X = hate_data['processed_text']
y = hate_data['hate_speech_binary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # Adjust test_size as needed

print("Training data shape:", X_train.shape, y_train.shape)
print("Testing data shape:", X_test.shape, y_test.shape)

Training data shape: (82159,) (82159,)
Testing data shape: (20540,) (20540,)


In [98]:
# prompt: Save the train and test

# Save the training and testing data to files (e.g., CSV)
train_data = pd.DataFrame({'text': X_train, 'labels': y_train})
test_data = pd.DataFrame({'text': X_test, 'labels': y_test})

train_data.to_csv('/content/drive/My Drive/EHR_PROJ/DATA/new_berkeley_train.csv', index=False)
test_data.to_csv('/content/drive/My Drive/EHR_PROJ/DATA/new_berkeley_test.csv', index=False)

print("Training data saved to:", '/content/drive/My Drive/EHR_PROJ/DATA/new_berkeley_train.csv')
print("Testing data saved to:", '/content/drive/My Drive/EHR_PROJ/DATA/new_berkeley_test.csv')

Training data saved to: /content/drive/My Drive/EHR_PROJ/DATA/new_berkeley_train.csv
Testing data saved to: /content/drive/My Drive/EHR_PROJ/DATA/new_berkeley_test.csv


In [99]:
hate_data['hate_speech_binary'].value_counts(normalize=True)

,proportion
hate_speech_binary,
0.0,0.52241
1.0,0.47759
